# Equation-Based SIR Model
## Lesson 3 · Section 1

In this notebook you will build and test the classical **SIR epidemic model** step by step.

We will:
1. Explain the meaning of the compartments `S`, `I`, and `R`
2. Translate the equations into Python functions
3. Test small parts of the model independently
4. Implement a simple Euler simulation
5. Visualise epidemic curves
6. Explore how the parameters change the outbreak
7. End with ideas for further independent work

The notebook is designed to be **educational**: each section contains small pieces that you can run and modify on your own.


## 0 · Imports
We only use NumPy and Matplotlib so the notebook stays simple and transparent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('Imports loaded successfully ✓')


---
## 1 · The SIR idea

The classical SIR model divides the population into three groups:

- `S` = susceptible individuals
- `I` = infected and infectious individuals
- `R` = recovered individuals

The model equations are:

$$\frac{dS}{dt} = -\beta S I$$
$$\frac{dI}{dt} = \beta S I - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

where:

- `β` is the infection rate
- `γ` is the recovery rate

### Interpretation
- New infections move people from `S` to `I`
- Recoveries move people from `I` to `R`
- The total population should remain constant: `S + I + R = N`


## 2 · Parameters and initial conditions
Keep all important values in one place so you can easily experiment later.


In [ ]:
beta = 0.35
gamma = 0.10

S0 = 0.99
I0 = 0.01
R0 = 0.00

T = 160
dt = 0.1

R_basic = beta / gamma

print(f'beta = {beta}')
print(f'gamma = {gamma}')
print(f'Basic reproduction number R0 = beta/gamma = {R_basic:.2f}')
print(f'Initial total population = {S0 + I0 + R0:.2f}')


---
## 3 · Write the differential equations as a Python function
This function does not simulate the epidemic yet. It only computes the **instantaneous rate of change** for one state `(S, I, R)`.

This is an important teaching step: first understand the model locally, then simulate it over time.


In [ ]:
def sir_rhs(S, I, R, beta, gamma):
    """Return the rates of change dS/dt, dI/dt, dR/dt."""
    dS = -beta * S * I
    dI = beta * S * I - gamma * I
    dR = gamma * I
    return dS, dI, dR

example = sir_rhs(S=0.99, I=0.01, R=0.0, beta=beta, gamma=gamma)
print('Rates for the initial state:', example)


### Independent test 1
Try a simple consistency check. If nobody is infected (`I = 0`), then nothing should change.


In [ ]:
test_result = sir_rhs(S=1.0, I=0.0, R=0.0, beta=0.5, gamma=0.2)
print('Test result:', test_result)
assert test_result == (-0.0, 0.0, 0.0)
print('Test passed ✓ No infected people means no epidemic dynamics.')


---
## 4 · One Euler step
The Euler method approximates the continuous system by small jumps in time:

$$X_{t+\Delta t} \approx X_t + \Delta t \cdot \frac{dX}{dt}$$

We first implement **one step only**, because this is easier to understand and test than writing the whole simulation at once.


In [ ]:
def euler_step_sir(S, I, R, beta, gamma, dt):
    dS, dI, dR = sir_rhs(S, I, R, beta, gamma)
    S_new = S + dt * dS
    I_new = I + dt * dI
    R_new = R + dt * dR
    return S_new, I_new, R_new

S1, I1, R1 = euler_step_sir(S0, I0, R0, beta, gamma, dt=1.0)
print(f'After one step: S={S1:.4f}, I={I1:.4f}, R={R1:.4f}')


### Independent test 2
Check that one Euler step still approximately preserves the total population.


In [ ]:
before = S0 + I0 + R0
after = S1 + I1 + R1
print(f'Total before = {before:.6f}')
print(f'Total after  = {after:.6f}')
assert np.isclose(before, after)
print('Test passed ✓ Population is conserved in this step.')


---
## 5 · Full simulation loop
Now we repeat the Euler step many times and store the full epidemic history.


In [ ]:
def simulate_sir_euler(S0, I0, R0, beta, gamma, T, dt):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S = np.zeros(steps)
    I = np.zeros(steps)
    R = np.zeros(steps)

    S[0], I[0], R[0] = S0, I0, R0

    for k in range(1, steps):
        S[k], I[k], R[k] = euler_step_sir(S[k - 1], I[k - 1], R[k - 1], beta, gamma, dt)

    return t, S, I, R

t, S, I, R = simulate_sir_euler(S0, I0, R0, beta, gamma, T, dt)
print(f'Simulation finished with {len(t)} stored time points.')


### Independent test 3
Check that the whole simulation conserves the total population over time.


In [ ]:
total_population = S + I + R
print('Minimum total:', total_population.min())
print('Maximum total:', total_population.max())
assert np.allclose(total_population, total_population[0])
print('Test passed ✓ The total population remains constant throughout the simulation.')


---
## 6 · Plot the epidemic curves
This is the classical SIR output: susceptible falls, infected rises and then falls, recovered increases.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, S, label='Susceptible', color='green', linewidth=2)
ax.plot(t, I, label='Infected', color='red', linewidth=2)
ax.plot(t, R, label='Recovered', color='blue', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Population fraction')
ax.set_title('Equation-Based SIR Model')
ax.legend()
plt.tight_layout()
plt.show()

peak_index = np.argmax(I)
print(f'Peak infected fraction = {I[peak_index]:.3f} at time t = {t[peak_index]:.1f}')


## 7 · A small guided experiment
The parameter ratio `beta/gamma` is very important. If it is large, the disease spreads more easily.

Try the prepared values below and compare the curves.


In [ ]:
parameter_sets = [
    {'beta': 0.15, 'gamma': 0.10, 'label': 'mild spread'},
    {'beta': 0.35, 'gamma': 0.10, 'label': 'baseline'},
    {'beta': 0.60, 'gamma': 0.10, 'label': 'fast spread'},
]

fig, axes = plt.subplots(len(parameter_sets), 1, figsize=(10, 10), sharex=True)

for ax, params in zip(axes, parameter_sets):
    t_tmp, S_tmp, I_tmp, R_tmp = simulate_sir_euler(
        S0, I0, R0, params['beta'], params['gamma'], T=120, dt=0.1
    )
    ax.plot(t_tmp, S_tmp, color='green', label='S')
    ax.plot(t_tmp, I_tmp, color='red', label='I')
    ax.plot(t_tmp, R_tmp, color='blue', label='R')
    ax.set_title(
        f"{params['label']} · beta={params['beta']} gamma={params['gamma']} R0={params['beta'] / params['gamma']:.2f}"
    )
    ax.set_ylabel('Fraction')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()


## 8 · What to try next
Run the previous cells again after changing one parameter at a time.

Suggested experiments:

- Increase `beta` and observe what happens to the infection peak
- Increase `gamma` and observe whether the outbreak ends faster
- Change `I0` from `0.01` to `0.10` and compare the start of the epidemic
- Change `dt` from `0.1` to `0.5` and discuss numerical accuracy
- Try `beta < gamma` and check whether the epidemic grows strongly or not

### Write your own conclusions
For each experiment, answer:
1. What changed in the graph?
2. Did the infection peak become higher or lower?
3. Did the epidemic last longer or shorter?
4. Which parameter had the strongest effect?


---
## 9 · Independent work and mini-project ideas
Choose one or more of the following projects for independent work.

### Project A · Add births and deaths
Extend the model so the total population is no longer constant.

### Project B · Add vaccination
Move part of the susceptible population directly to `R` at the beginning or during the simulation.

### Project C · Compare SIR and SIS
Modify the recovery rule so recovered individuals become susceptible again.

### Project D · Compute useful summary values
Write code that automatically reports:
- time of infection peak
- size of infection peak
- final recovered fraction

### Project E · Explore numerical methods
Implement a second-order method and compare it with Euler.

### Final reflection
After finishing, explain in your own words:
- What assumptions the equation-based SIR model makes
- What the model can explain well
- What the model cannot capture well
- Why an agent-based approach might be useful in some situations
